In [1]:
import pandas as pd

df = pd.read_csv('datasets/favorita_sales.csv')
df.info()

/var/folders/5l/17wd4myj387b056cpgkkpxth0000gn/T/ipykernel_26173/942700094.py:3: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('datasets/favorita_sales.csv')


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 125497040 entries, 0 to 125497039
Data columns (total 16 columns):
 #   Column        Dtype  
---  ------        -----  
 0   id            int64  
 1   date          object 
 2   store_nbr     int64  
 3   item_nbr      int64  
 4   unit_sales    float64
 5   onpromotion   object 
 6   family        object 
 7   class         int64  
 8   perishable    int64  
 9   city          object 
 10  state         object 
 11  type          object 
 12  cluster       int64  
 13  dcoilwtico    float64
 14  transactions  float64
 15  is_holiday    float64
dtypes: float64(4), int64(6), object(6)
memory usage: 15.0+ GB


In [3]:
import numpy as np

def downcast_dataframe(df):
    """Safely downcast int and float columns to reduce memory usage."""
    
    original_memory = df.memory_usage(deep=True).sum() / 1024**3
    print(f"Memory before: {original_memory:.2f} GB")

    # Downcast integers
    int_cols = df.select_dtypes(include=['int64', 'int32']).columns
    for col in int_cols:
        col_min, col_max = df[col].min(), df[col].max()
        
        if col_min >= np.iinfo(np.int8).min and col_max <= np.iinfo(np.int8).max:
            df[col] = df[col].astype(np.int8)
        elif col_min >= np.iinfo(np.int16).min and col_max <= np.iinfo(np.int16).max:
            df[col] = df[col].astype(np.int16)
        elif col_min >= np.iinfo(np.int32).min and col_max <= np.iinfo(np.int32).max:
            df[col] = df[col].astype(np.int32)
        # else: keep as int64

    # Downcast floats
    float_cols = df.select_dtypes(include=['float64']).columns
    for col in float_cols:
        downcast = pd.to_numeric(df[col], downcast='float')  # tries float32 first
        # Verify no data loss beyond float32 precision tolerance
        if np.allclose(df[col].dropna(), downcast.dropna(), rtol=1e-5, equal_nan=True):
            df[col] = downcast
        # else: keep as float64

    new_memory = df.memory_usage(deep=True).sum() / 1024**3
    print(f"Memory after:  {new_memory:.2f} GB")
    print(f"Reduction:     {(1 - new_memory/original_memory)*100:.1f}%")
    
    print("\nColumn dtypes after downcasting:")
    print(df.dtypes)
    
    return df



In [5]:
df = downcast_dataframe(df)

Memory before: 46.22 GB
Memory after:  40.26 GB
Reduction:     12.9%

Column dtypes after downcasting:
id                int32
date             object
store_nbr          int8
item_nbr          int32
unit_sales      float32
onpromotion      object
family           object
class             int16
perishable         int8
city             object
state            object
type             object
cluster            int8
dcoilwtico      float32
transactions    float32
is_holiday      float32
dtype: object


In [8]:
cols_to_keep = [
    "date",
    "store_nbr",
    "item_nbr",
    "unit_sales",
    "cluster",
    "state",
    "dcoilwtico",
    "is_holiday"
]

df = df[cols_to_keep]

In [9]:
df.info(show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 125497040 entries, 0 to 125497039
Data columns (total 8 columns):
 #   Column      Non-Null Count      Dtype  
---  ------      --------------      -----  
 0   date        125497040 non-null  object 
 1   store_nbr   125497040 non-null  int8   
 2   item_nbr    125497040 non-null  int32  
 3   unit_sales  125497040 non-null  float32
 4   cluster     125497040 non-null  int8   
 5   state       125497040 non-null  object 
 6   dcoilwtico  125496462 non-null  float32
 7   is_holiday  125497040 non-null  float32
dtypes: float32(3), int32(1), int8(2), object(2)
memory usage: 4.0+ GB


In [10]:
df["date"] = pd.to_datetime(df["date"])
df.info(show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 125497040 entries, 0 to 125497039
Data columns (total 8 columns):
 #   Column      Non-Null Count      Dtype         
---  ------      --------------      -----         
 0   date        125497040 non-null  datetime64[ns]
 1   store_nbr   125497040 non-null  int8          
 2   item_nbr    125497040 non-null  int32         
 3   unit_sales  125497040 non-null  float32       
 4   cluster     125497040 non-null  int8          
 5   state       125497040 non-null  object        
 6   dcoilwtico  125496462 non-null  float32       
 7   is_holiday  125497040 non-null  float32       
dtypes: datetime64[ns](1), float32(3), int32(1), int8(2), object(1)
memory usage: 4.0+ GB


In [13]:
print(df["date"].min())
print(df["date"].max())

2013-01-01 00:00:00
2017-08-15 00:00:00


In [14]:
smaller_df = df[df["date"] >= "2015-08-15"]
smaller_df.info(show_counts=True)

<class 'pandas.core.frame.DataFrame'>
Index: 71541001 entries, 53956039 to 125497039
Data columns (total 8 columns):
 #   Column      Non-Null Count     Dtype         
---  ------      --------------     -----         
 0   date        71541001 non-null  datetime64[ns]
 1   store_nbr   71541001 non-null  int8          
 2   item_nbr    71541001 non-null  int32         
 3   unit_sales  71541001 non-null  float32       
 4   cluster     71541001 non-null  int8          
 5   state       71541001 non-null  object        
 6   dcoilwtico  71541001 non-null  float32       
 7   is_holiday  71541001 non-null  float32       
dtypes: datetime64[ns](1), float32(3), int32(1), int8(2), object(1)
memory usage: 2.8+ GB


In [15]:
smaller_df = smaller_df.drop(columns=['dcoilwtico'])
smaller_df.info(show_counts=True)

<class 'pandas.core.frame.DataFrame'>
Index: 71541001 entries, 53956039 to 125497039
Data columns (total 7 columns):
 #   Column      Non-Null Count     Dtype         
---  ------      --------------     -----         
 0   date        71541001 non-null  datetime64[ns]
 1   store_nbr   71541001 non-null  int8          
 2   item_nbr    71541001 non-null  int32         
 3   unit_sales  71541001 non-null  float32       
 4   cluster     71541001 non-null  int8          
 5   state       71541001 non-null  object        
 6   is_holiday  71541001 non-null  float32       
dtypes: datetime64[ns](1), float32(2), int32(1), int8(2), object(1)
memory usage: 2.5+ GB


In [16]:
smaller_df = smaller_df[smaller_df["date"] >= "2016-01-01"]

In [17]:
smaller_df.info(show_counts=True)

<class 'pandas.core.frame.DataFrame'>
Index: 59038132 entries, 66458908 to 125497039
Data columns (total 7 columns):
 #   Column      Non-Null Count     Dtype         
---  ------      --------------     -----         
 0   date        59038132 non-null  datetime64[ns]
 1   store_nbr   59038132 non-null  int8          
 2   item_nbr    59038132 non-null  int32         
 3   unit_sales  59038132 non-null  float32       
 4   cluster     59038132 non-null  int8          
 5   state       59038132 non-null  object        
 6   is_holiday  59038132 non-null  float32       
dtypes: datetime64[ns](1), float32(2), int32(1), int8(2), object(1)
memory usage: 2.1+ GB


In [19]:
smaller_df.head()

,date,store_nbr,item_nbr,unit_sales,cluster,state,is_holiday
66458908,2016-01-01,25,105574,12.0,1,Santa Elena,1.0
66458909,2016-01-01,25,105575,9.0,1,Santa Elena,1.0
66458910,2016-01-01,25,105857,3.0,1,Santa Elena,1.0
66458911,2016-01-01,25,108634,3.0,1,Santa Elena,1.0
66458912,2016-01-01,25,108701,2.0,1,Santa Elena,1.0


In [20]:
smaller_df.to_csv('sample_dataset.csv')

In [ ]:
import pandas as pd 
df = pd.read_csv("datasets/sample_dataset.csv", parse_dates=['date'])
df.info(show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 59038132 entries, 0 to 59038131
Data columns (total 8 columns):
 #   Column      Non-Null Count     Dtype         
---  ------      --------------     -----         
 0   Unnamed: 0  59038132 non-null  int64         
 1   date        59038132 non-null  datetime64[ns]
 2   store_nbr   59038132 non-null  int64         
 3   item_nbr    59038132 non-null  int64         
 4   unit_sales  59038132 non-null  float64       
 5   cluster     59038132 non-null  int64         
 6   state       59038132 non-null  object        
 7   is_holiday  59038132 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(4), object(1)
memory usage: 3.5+ GB


In [4]:
df = downcast_dataframe(df)

Memory before: 6.26 GB
Memory after:  4.61 GB
Reduction:     26.3%

Column dtypes after downcasting:
Unnamed: 0             int32
date          datetime64[ns]
store_nbr               int8
item_nbr               int32
unit_sales           float32
cluster                 int8
state                 object
is_holiday           float32
dtype: object


In [5]:
df.info(show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 59038132 entries, 0 to 59038131
Data columns (total 8 columns):
 #   Column      Non-Null Count     Dtype         
---  ------      --------------     -----         
 0   Unnamed: 0  59038132 non-null  int32         
 1   date        59038132 non-null  datetime64[ns]
 2   store_nbr   59038132 non-null  int8          
 3   item_nbr    59038132 non-null  int32         
 4   unit_sales  59038132 non-null  float32       
 5   cluster     59038132 non-null  int8          
 6   state       59038132 non-null  object        
 7   is_holiday  59038132 non-null  float32       
dtypes: datetime64[ns](1), float32(2), int32(2), int8(2), object(1)
memory usage: 1.9+ GB


In [6]:
df.head()

,Unnamed: 0,date,store_nbr,item_nbr,unit_sales,cluster,state,is_holiday
0,66458908,2016-01-01,25,105574,12.0,1,Santa Elena,1.0
1,66458909,2016-01-01,25,105575,9.0,1,Santa Elena,1.0
2,66458910,2016-01-01,25,105857,3.0,1,Santa Elena,1.0
3,66458911,2016-01-01,25,108634,3.0,1,Santa Elena,1.0
4,66458912,2016-01-01,25,108701,2.0,1,Santa Elena,1.0


In [7]:
print(df["date"].min())
print(df["date"].max())

2016-01-01 00:00:00
2017-08-15 00:00:00


In [8]:
df = df[df["date"] >= '2017-01-01']
df.info(show_counts=True)

<class 'pandas.core.frame.DataFrame'>
Index: 23808261 entries, 35229871 to 59038131
Data columns (total 8 columns):
 #   Column      Non-Null Count     Dtype         
---  ------      --------------     -----         
 0   Unnamed: 0  23808261 non-null  int32         
 1   date        23808261 non-null  datetime64[ns]
 2   store_nbr   23808261 non-null  int8          
 3   item_nbr    23808261 non-null  int32         
 4   unit_sales  23808261 non-null  float32       
 5   cluster     23808261 non-null  int8          
 6   state       23808261 non-null  object        
 7   is_holiday  23808261 non-null  float32       
dtypes: datetime64[ns](1), float32(2), int32(2), int8(2), object(1)
memory usage: 953.6+ MB


In [9]:
original_memory = df.memory_usage(deep=True).sum() / 1024**3
print(f"Memory now: {original_memory:.2f} GB")

Memory now: 2.04 GB


In [14]:
df.to_parquet('datasets/sample_dataset.parquet', index=False)

In [1]:
import pandas as pd
df = pd.read_parquet('datasets/sample_dataset.parquet')

In [2]:
df.head()

,Unnamed: 0,date,store_nbr,item_nbr,unit_sales,cluster,state,is_holiday
0,101688779,2017-01-01,25,99197,1.0,1,Santa Elena,0.0
1,101688780,2017-01-01,25,103665,7.0,1,Santa Elena,0.0
2,101688781,2017-01-01,25,105574,1.0,1,Santa Elena,0.0
3,101688782,2017-01-01,25,105857,4.0,1,Santa Elena,0.0
4,101688783,2017-01-01,25,106716,2.0,1,Santa Elena,0.0


In [4]:
df.info(show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23808261 entries, 0 to 23808260
Data columns (total 8 columns):
 #   Column      Non-Null Count     Dtype         
---  ------      --------------     -----         
 0   Unnamed: 0  23808261 non-null  int32         
 1   date        23808261 non-null  datetime64[ns]
 2   store_nbr   23808261 non-null  int8          
 3   item_nbr    23808261 non-null  int32         
 4   unit_sales  23808261 non-null  float32       
 5   cluster     23808261 non-null  int8          
 6   state       23808261 non-null  object        
 7   is_holiday  23808261 non-null  float32       
dtypes: datetime64[ns](1), float32(2), int32(2), int8(2), object(1)
memory usage: 772.0+ MB


In [7]:
df = df.drop(columns=["Unnamed: 0"])
df.head()

,date,store_nbr,item_nbr,unit_sales,cluster,state,is_holiday
0,2017-01-01,25,99197,1.0,1,Santa Elena,0.0
1,2017-01-01,25,103665,7.0,1,Santa Elena,0.0
2,2017-01-01,25,105574,1.0,1,Santa Elena,0.0
3,2017-01-01,25,105857,4.0,1,Santa Elena,0.0
4,2017-01-01,25,106716,2.0,1,Santa Elena,0.0


In [8]:
df.to_parquet('datasets/sample_dataset.parquet', index=False)